In [0]:
%run "/Workspace/Users/jeevan.azureacc3@gmail.com/bankaml-de-project/notebooks/04_utils/watermark_incremental_load"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
bronze_customers_df = spark.read.format("delta").table("bankaml.bronze.customers")
bronze_customers_df = get_new_rows_from_source_df(bronze_customers_df, "silver", "silver", "customers")

customers_quarantine_df = bronze_customers_df.filter(col("customer_id").isNull() | col("name").isNull())

bronze_customers_df = bronze_customers_df.filter(col("customer_id").isNotNull() | col("name").isNotNull()) \
    .dropDuplicates(["customer_id"])

## Data type conversion
customers_df = bronze_customers_df.withColumns(
    {
        "created_at": col("created_at").cast(TimestampType()),
    }
)

## Adding Hash and Audit columns
customers_df = customers_df.withColumns(
    {
        "country": upper(col("country")),
        "needs_review": when(col("kyc_status").isNull() | col("risk_rating").isNull(), lit("Y").cast(StringType())),
        "effective_start_date": col("created_at"),
        "effective_end_date": lit("9999-12-31").cast(TimestampType()),
        "is_current": lit("Y").cast(StringType()),
        "hash_value": 
            sha2(
                concat_ws(
                    "||",
                coalesce(col("risk_rating"), lit("")),
                coalesce(col("kyc_status"), lit(""))
            )
            , 256
        ),
        "updated_at": lit(None).cast(TimestampType()),
        "updated_by": lit(None).cast(StringType())
    }
)

In [0]:
%sql
create table if not exists bankaml.silver.customers
(
    customer_id string,
    name string,
    dob date,
    ssn_hash string,
    risk_rating string,
    kyc_status string, 
    country string,
    needs_review string,
    effective_start_date timestamp,
    effective_end_date timestamp,
    is_current string,
    created_at timestamp,
    created_by string,
    hash_value string,
    updated_at timestamp,
    updated_by string
)
using delta;

In [0]:
customers_target_table = DeltaTable.forName(spark, "bankaml.silver.customers")
target_df = customers_target_table.toDF()
customers_df = customers_df.select(*target_df.columns)

joined_df = (
    customers_df.alias("s").join(target_df.alias("t"), col("s.customer_id") == col("t.customer_id"), "left")
)

new_customers_df = joined_df.filter(col("t.customer_id").isNull()).select("s.*")
existing_customers_df = joined_df.filter(col("t.customer_id").isNotNull()).select("s.*")

In [0]:
## Existing customer rows: Need to expire the existing row if hash_value changes and then insertion of updated row.
(
    customers_target_table.alias("t").merge(
        existing_customers_df.alias("s"), 
        """
        t.customer_id=s.customer_id
        and t.is_current = "Y"
        """
    )
    .whenMatchedUpdate(
        condition= "t.hash_value <> s.hash_value",
        set = {
            "effective_end_date": "s.created_at",
            "is_current": lit("N"),
            "updated_at": "current_timestamp()",
            "updated_by": lit("Databricks-silver-customers")
        }
    )
    .execute()
)

In [0]:
## Exsiting customer insertion if the hash_value changed
active_customers_df = target_df.filter(col("is_current") == "Y")
changed_customers_df = (
    existing_customers_df.alias("s").join(
        active_customers_df.alias("t"), 
        "customer_id", 
        "inner"
    )
    .filter(col("s.hash_value") != col("t.hash_value"))
    .select("s.*")
)

In [0]:
## New customer rows & Changed customers rows insertion
insertion_df = changed_customers_df.unionByName(new_customers_df)
insertion_df.write.format("delta").mode("append").saveAsTable("bankaml.silver.customers")

update_last_processed_value(bronze_customers_df, "silver", "silver", "customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankaml.quarantine.customers (
    customer_id string,
    name string,
    dob STRING,
    ssn_hash string,
    risk_rating string,
    kyc_status string, 
    country string,
    created_at STRING,
    created_by STRING,
    _ingestion_ts TIMESTAMP,
    quarantine_reason STRING,  
    quarantined_at TIMESTAMP,
    source_layer STRING
)
using delta;

In [0]:
customers_quarantine_df = customers_quarantine_df.withColumns(
    {
        "quarantine_reason": 
            when((col("customer_id").isNull()) & (col("name").isNull()), lit("customer_id and name are Null"))
            .when(col("customer_id").isNull(), lit("customer_id is null"))
            .when(col("name").isNull(), lit("name is null"))
            .otherwise(lit("unknown")),
        "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)
customers_quarantine_df.write.format("delta").mode("append").saveAsTable("bankaml.quarantine.customers")